# H-001 — Kanal Konumu (Range / Mean-Reversion)

**Hipotez:** Fiyat, son N barlık kanalın dibine yakınken ileriye dönük getirisi
ortalamanın üstünde; tepesine yakınken altındadır.

---

## Çalıştırmadan önce

1. `hypotheses/H-001-*.md` kaydı **yazıldı ve commit'lendi** mi?
   Sonucu görüp kriter yazmak tüm disiplini çökertir.
2. Aşağıdaki **6 karar** dolduruldu mu? Hepsi sonuç görülmeden sabitlenir.
3. Bu defter yalnızca **araştırma alanında** çalışır. Kasa (2024-07 sonrası)
   en sonda, bir kez, ayrı bir defterde açılır.


## 0. Kararlar

Değiştirdiğiniz her şey bir **deneme** sayılır ve `hypotheses/REGISTRY.md`'ye yazılır.


In [1]:
# --- KARAR 1: sembol -------------------------------------------------
SYMBOL = 'SPY'          # 'SPY' | 'AAPL'   (ikisi de olursa IKI deneme)

# --- KARAR 2: frekans ------------------------------------------------
TIMEFRAME = '1Day'      # '1Day' | '1Hour' (saatlikte PDT duvari - TD-04)

# --- KARAR 3: hedef ufku (K bar) -------------------------------------
K = 1                   # kac bar sonrasina bakiyoruz

# --- KARAR 4: pencereler (N) -----------------------------------------
WINDOWS = (20, 50, 100) # HEPSI bastan yazilir; sonradan ekleme = sayilmamis deneme

# --- KARAR 5: maliyet esigi ------------------------------------------
COST = 0.0005           # 5 baz puan. Bunu gecmeyen hareket 'yon' sayilmaz (TD-11)

# --- KARAR 6: basari kriteri -----------------------------------------
# Sonucu GORMEDEN doldurun. Bonferroni: 0.05 / deneme_sayisi
N_TRIALS = len(WINDOWS)     # + sembol / frekans varyasyonlari
ALPHA    = 0.05 / N_TRIALS
MIN_RHO  = 0.05             # bu buyuklugun altindaki etki anlamli olsa da ise yaramaz

print(f'{N_TRIALS} deneme -> duzeltilmis esik alpha = {ALPHA:.5f}')


3 deneme -> duzeltilmis esik alpha = 0.01667


## 1. Veri ve bölme


In [2]:
import sys; sys.path.insert(0, '..')
import pandas as pd

from src.data.load import load_bars
from lab.splits import (split_research_vault, iter_subperiods,
                        iter_stress_windows, purged_walk_forward)
from lab.h001 import build_table
from lab.analysis import rank_correlation, baseline, bin_table, subperiod_report

pd.set_option('display.float_format', lambda v: f'{v:,.4f}')

df = load_bars(SYMBOL, TIMEFRAME)
split = split_research_vault(df, horizon_bars=K)
split


Split(research=2135 bar [2016-01-04 → 2024-06-27], vault=548 bar [2024-07-01 → 2026-09-04], purged=1, horizon=1)

**Kasa buradan sonra hiç kullanılmayacak.** `split.research` ile çalışıyoruz.


In [3]:
table = build_table(split.research, windows=WINDOWS, k=K, cost=COST)
print(f'{len(table)} satir, {table.attrs["dropped"]} satir dustu (isinma + cevapsiz kuyruk)')
table.head()


2035 satir, 100 satir dustu (isinma + cevapsiz kuyruk)


,timestamp,close,kanal_konumu_20,kanal_konumu_50,kanal_konumu_100,ileri_getiri_1,ileri_yon_1
0,2016-05-25 04:00:00+00:00,178.1300,0.9294,0.8246,0.9450,0.0003,0.0000
1,2016-05-26 04:00:00+00:00,178.1800,0.9378,0.8274,0.9470,0.0043,1.0000
2,2016-05-27 04:00:00+00:00,178.9500,0.9984,0.9258,0.9772,-0.0019,0.0000
3,2016-05-31 04:00:00+00:00,178.6100,0.8930,0.8824,0.9639,0.0020,1.0000
4,2016-06-01 04:00:00+00:00,178.9700,0.9465,0.9284,0.9780,0.0031,1.0000


## 2. Baseline — hiçbir şey bilmeseydik

Her sonucun yanında bu durmalı. "%54 yukarı" tek başına anlamsızdır;
taban %53 ise hipotez bir şey söylemiyordur.


In [4]:
baseline(table[f'ileri_getiri_{K}'], cost=COST)


bar               2,035.0000
ortalama_getiri       0.0006
medyan_getiri         0.0007
yukari_orani          0.5174
std                   0.0112
dtype: float64

## 3. Birincil istatistik — sıra korelasyonu

Dilim/eşik seçimi içermez, dolayısıyla "en iyi dilimi seç" serbestliği yoktur.

**Mean-reversion hipotezi NEGATİF rho bekler:** kanal konumu arttıkça
(tepeye yaklaştıkça) ileri getiri azalmalı.


In [5]:
rows = []
for n in WINDOWS:
    res = rank_correlation(table[f'kanal_konumu_{n}'],
                           table[f'ileri_getiri_{K}'], horizon_bars=K)
    rows.append({'N': n, 'rho': res.rho, 'p': res.p_value,
                 'n_eff': res.effective_n,
                 'gecti_mi': (res.p_value < ALPHA) and (abs(res.rho) >= MIN_RHO)})
pd.DataFrame(rows)


,N,rho,p,n_eff,gecti_mi
0,20,-0.0406,0.0669,2035,False
1,50,-0.0458,0.0389,2035,False
2,100,-0.0383,0.0837,2035,False


## 4. Dilim tablosu — yalnızca gözle görmek için

Buradaki en iyi dilime bakıp karar vermek **p-hacking**'dir.
Karar yukarıdaki rho ile verildi. Bu tablo ilişkinin **monotonik** olup
olmadığını görmek için: dipten tepeye düzenli bir eğim var mı?


In [6]:
N_SHOW = WINDOWS[0]
bin_table(table[f'kanal_konumu_{N_SHOW}'], table[f'ileri_getiri_{K}'], cost=COST)


,bar,ortalama_getiri,medyan_getiri,yukari_orani,std
dilim,,,,,
"(-0.001, 0.1]",119,0.0023,0.0004,0.4790,0.0194
"(0.1, 0.2]",81,0.0010,0.0046,0.5556,0.0174
"(0.2, 0.3]",105,-0.0014,0.0002,0.4762,0.0198
"(0.3, 0.4]",100,0.0010,0.0024,0.5400,0.0145
"(0.4, 0.5]",105,0.0014,0.0041,0.6190,0.0148
"(0.5, 0.6]",152,-0.0007,-0.0008,0.4671,0.0113
"(0.6, 0.7]",166,0.0008,0.0009,0.5241,0.0109
"(0.7, 0.8]",178,0.0012,0.0011,0.5449,0.0082
"(0.8, 0.9]",338,0.0012,0.0012,0.5680,0.0073


## 5. Alt dönem tutarlılığı

Etki üç rejimde de aynı işareti taşıyor mu? Tek dönemde çıkan etki
o rejime özgüdür — genel bir yasa değildir.


In [7]:
subperiod_report(table,
                 feature_col=f'kanal_konumu_{WINDOWS[0]}',
                 target_col=f'ileri_getiri_{K}',
                 horizon_bars=K,
                 subperiods=iter_subperiods(table))


,donem,bar,rho,p,not
0,2016-2019 sakin/yükseliş,907,-0.0565,0.0887,
1,2020-2021 covid + toparlanma,505,-0.1174,0.0080,
2,2022-2024H1 ayı + faiz şoku,623,0.0107,0.7899,


## 6. Stres penceresi raporu

**Seçim kriteri değil, risk bilgisi.** Kasa kriz içermiyor, dolayısıyla
stratejinin çöküşte ne yaptığını yalnızca buradan öğrenebiliriz.
Soru "kazandı mı" değil: **ne kadar kaybetti, dayanabilir miyiz?**


In [8]:
for name, part in iter_stress_windows(table):
    if len(part) < 30:
        print(f'{name}: {len(part)} bar - cok az')
        continue
    b = baseline(part[f'ileri_getiri_{K}'], cost=COST)
    res = rank_correlation(part[f'kanal_konumu_{WINDOWS[0]}'],
                           part[f'ileri_getiri_{K}'], horizon_bars=K)
    print(f'{name}: {len(part)} bar  rho={res.rho:+.3f}  '
          f'ort_getiri={b["ortalama_getiri"]:+.4f}')


covid çöküşü: 51 bar  rho=+0.016  ort_getiri=-0.0026
2022 ayı piyasası: 209 bar  rho=-0.023  ort_getiri=-0.0009


## 7. Walk-forward — etki katlar arasında tutarlı mı

Tek bir toplam sayı dönemsel bir tesadüfü gizleyebilir. Katların
çoğunda aynı işaret görünmüyorsa etki kararlı değildir.


In [9]:
for i, (tr, te) in enumerate(purged_walk_forward(len(table), horizon_bars=K), 1):
    block = table.iloc[te]
    res = rank_correlation(block[f'kanal_konumu_{WINDOWS[0]}'],
                           block[f'ileri_getiri_{K}'], horizon_bars=K)
    print(f'kat {i}: test {len(te):>4} bar  rho={res.rho:+.4f}  p={res.p_value:.4f}')


kat 1: test  339 bar  rho=-0.0331  p=0.5426
kat 2: test  339 bar  rho=-0.0871  p=0.1085
kat 3: test  339 bar  rho=-0.1463  p=0.0066
kat 4: test  339 bar  rho=-0.0069  p=0.8999
kat 5: test  340 bar  rho=-0.0174  p=0.7484


---

## 8. Sonucu kaydet

Sonuç ne olursa olsun `hypotheses/H-001-*.md` dosyasına yazılır ve
`REGISTRY.md`'de deneme sayacı güncellenir. **Reddedilen hipotez silinmez** —
kaç deneme yaptığımızı bilmek, kalanların anlamlılığını belirliyor (TD-14).

### KASA

`split.vault` bu defterde **açılmaz**. Kasa yalnızca tüm hipotezler seçilip
nihai birleşik sistem kurulduktan sonra, ayrı bir defterde, **bir kez** açılır.
